# E-Commerce Sales & Customer Analytics
## Import Olist CSV Dataset into MySQL

This notebook imports the 9 Olist CSV files into the `e_commerce_sales` MySQL database and verifies the imported row counts.

### Project workflow
1. Install/import required Python libraries
2. Configure MySQL connection
3. Locate the local Dataset folder
4. Load the 9 CSV files
5. Import them into MySQL
6. Verify row counts


In [ ]:
# Run this once if the libraries are not installed
# !pip install pandas sqlalchemy pymysql

In [ ]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

print('Libraries loaded successfully.')

## 1. MySQL connection

Replace `YOUR_PASSWORD` with your MySQL password.

In [ ]:
MYSQL_USER = 'root'
MYSQL_PASSWORD = 'YOUR_PASSWORD'
MYSQL_HOST = 'localhost'
MYSQL_PORT = 3306
DATABASE = 'e_commerce_sales'

engine = create_engine(
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{DATABASE}'
)

with engine.connect() as connection:
    print('MySQL connection successful!')

## 2. Dataset location

Keep the 9 CSV files inside your local `Dataset` folder.

In [ ]:
DATASET_PATH = Path('../Dataset')

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset folder not found: {DATASET_PATH.resolve()}\n'
        'Update DATASET_PATH to the location of your Dataset folder.'
    )

print('Dataset folder:', DATASET_PATH.resolve())

## 3. Olist CSV → MySQL table mapping

In [ ]:
FILES = {
    'olist_customers_dataset.csv': 'customers',
    'olist_orders_dataset.csv': 'orders',
    'olist_order_items_dataset.csv': 'order_items',
    'olist_products_dataset.csv': 'products',
    'olist_sellers_dataset.csv': 'sellers',
    'olist_order_payments_dataset.csv': 'payments',
    'olist_order_reviews_dataset.csv': 'reviews',
    'olist_geolocation_dataset.csv': 'geolocations',
    'product_category_name_translation.csv': 'category_translation'
}

print(f'{len(FILES)} CSV files configured for import.')

## 4. Check that all CSV files exist

In [ ]:
missing_files = []

for csv_file in FILES:
    if not (DATASET_PATH / csv_file).exists():
        missing_files.append(csv_file)

if missing_files:
    print('Missing files:')
    for file in missing_files:
        print(' -', file)
else:
    print('All 9 CSV files found successfully!')

## 5. Import CSV files into MySQL

The notebook imports each CSV as a MySQL table using the table names defined above.

In [ ]:
import_results = []

for csv_file, table_name in FILES.items():
    file_path = DATASET_PATH / csv_file
    print(f'Loading {csv_file}...')

    df = pd.read_csv(file_path)
    df.to_sql(
        table_name,
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=5000,
        method='multi'
    )

    import_results.append({
        'table': table_name,
        'rows_imported': len(df),
        'columns': len(df.columns)
    })

    print(f'  ✓ {table_name}: {len(df):,} rows imported')

print('\nAll files imported successfully!')

## 6. Verify MySQL row counts

In [ ]:
verification = []

with engine.connect() as connection:
    for table_name in FILES.values():
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM `{table_name}`')
        )
        count = result.scalar()
        verification.append({'table': table_name, 'mysql_rows': count})

verification_df = pd.DataFrame(verification)
verification_df

## 7. Final status

In [ ]:
print('========================================')
print(' E-COMMERCE DATA IMPORT COMPLETED')
print('========================================')
print(f'Tables imported: {len(FILES)}')
print('Database:', DATABASE)
print('Status: SUCCESS')